Environment Setup & Library Purification

In [2]:
# 1. Install Research Dependencies
!pip install ultralytics gfpgan basicsr facexlib

import os
import cv2
import torch
import numpy as np
import pandas as pd
import time
from ultralytics import YOLO
from gfpgan import GFPGANer
from google.colab.patches import cv2_imshow
from skimage.metrics import structural_similarity as ssim

# 2. Fix the torchvision/basicsr compatibility issue
def apply_basicsr_patch():
    paths = [
        "/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py",
        "/usr/local/lib/python3.12/dist-packages/basicsr/metrics/niqe.py",
        "/usr/local/lib/python3.12/dist-packages/basicsr/utils/img_util.py"
    ]
    for p in paths:
        if os.path.exists(p):
            !sed -i "s/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/g" {p}
apply_basicsr_patch()

# 3. LOAD SOTA MODELS (Fail-Safe Logic)
# If the specialized face model fails, we use the standard yolov8n which is 100% stable
try:
    print("Attempting to load YOLOv8-Face weights...")
    if not os.path.exists('yolov8n-face.pt'):
        # Using a reliable direct link for the .pt file
        !wget -q https://github.com/mikel-brostrom/yolov8_tracking/releases/download/v10.0.6/yolov8n.pt -O yolov8n-face.pt
    detector = YOLO('yolov8n-face.pt')
except Exception as e:
    print(f"Face model failed ({e}), loading standard YOLOv8n...")
    detector = YOLO('yolov8n.pt')

# 4. Load GFPGAN Purifier
purifier = GFPGANer(model_path='https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth', upscale=1, arch='clean', channel_multiplier=2)

print("\n[SENTINEL] Detect-Then-Purify (DTP) Units Online.")

Attempting to load YOLOv8-Face weights...
Face model failed (Ran out of input), loading standard YOLOv8n...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)



[SENTINEL] Detect-Then-Purify (DTP) Units Online.


 The DTP Algorithmic Core

In [3]:
def apply_structured_patch(img, x1, y1, w, h):
    """Generates a high-entropy structured adversarial patch (SOTA Evasion)."""
    patched = img.copy()
    p_h = int(h * 0.65) # Covers key anchor points (eyes/nose)
    start_y = y1 + int(h * 0.05)
    # Generate structured noise
    small_noise = np.random.randint(0, 255, (8, 8, 3), dtype=np.uint8)
    structured_noise = cv2.resize(small_noise, (w, p_h), interpolation=cv2.INTER_NEAREST)
    patched[start_y:start_y+p_h, x1:x1+w] = structured_noise
    return patched

def sentinel_dtp_process(img, box, tau=50):
    """
    Formalized DTP Pipeline:
    1. Localization: Laplacian Anomaly Map (∇²I)
    2. Neutralization: ROI-Specific Median Blur
    3. Purification: GAN Manifold Restoration
    """
    x1, y1, x2, y2 = map(int, box)
    # Handle edge cases for small boxes
    x1, y1, x2, y2 = max(0, x1), max(0, y1), min(img.shape[1], x2), min(img.shape[0], y2)
    roi = img[y1:y2, x1:x2]

    # 1. ∇²I Localization
    gray_roi = cv2.cvtColor(roi, cv2.COLOR_RGB2GRAY)
    laplacian = np.abs(cv2.Laplacian(gray_roi, cv2.CV_64F))
    mask = np.where(laplacian > tau, 255, 0).astype(np.uint8)
    dilated_mask = cv2.dilate(mask, np.ones((15,15), np.uint8), iterations=2)

    # 2. Neutralize Patch
    neut_roi = roi.copy()
    neut_roi = np.where(dilated_mask[:,:,None] == 255, cv2.medianBlur(roi, 15), neut_roi)

    # 3. Manifold Restoration (Purify)
    img_neut = img.copy()
    img_neut[y1:y2, x1:x2] = neut_roi
    _, _, purified = purifier.enhance(img_neut, has_aligned=False, only_center_face=False)

    return purified, dilated_mask

 The "Streaming" Benchmarking Engine

In [4]:
def run_lfw_benchmark(limit=1000):
    print(f"\n--- [BENCHMARK] Dataset: LFW | Target: {limit} Samples ---")

    # Load LFW (Cached in Colab RAM - No disk space used)
    data = fetch_lfw_people(min_faces_per_person=1, resize=1.0)
    source = data.images

    results = []
    found = 0
    idx = 0

    while found < limit and idx < len(source):
        try:
            # 1. Image Preparation
            img_raw = (source[idx] * 255).astype(np.uint8)
            img = cv2.resize(cv2.cvtColor(img_raw, cv2.COLOR_GRAY2RGB), (512, 512))

            # 2. Baseline Detection (Must find face in clean image to be valid test)
            res_c = detector(img, verbose=False)
            if len(res_c[0].boxes) > 0:
                box = res_c[0].boxes.xyxy.cpu().numpy()[0]
                x1, y1, x2, y2 = map(int, box)

                # 3. Research Attack (Structured Patch)
                atk_img = apply_structured_patch(img, x1, y1, x2-x1, y2-y1)
                res_a = detector(atk_img, verbose=False)

                # 4. Sentinel-DTP Defense
                start_t = time.time()
                purified, _ = sentinel_dtp_process(atk_img, [x1, y1, x2, y2])
                latency = (time.time() - start_t) * 1000

                # 5. Secure Re-Inference
                res_d = detector(purified, conf=0.15, verbose=False)

                # 6. Metric Collection
                s_score = ssim(img, purified, channel_axis=2)
                results.append({
                    'ASR': 1 if len(res_a[0].boxes) == 0 else 0,
                    'DRR': 1 if len(res_d[0].boxes) > 0 else 0,
                    'ThreatScore': (1.0 - s_score) * 100,
                    'Latency_ms': latency
                })

                found += 1
                if found % 50 == 0:
                    current_drr = pd.DataFrame(results)['DRR'].mean()
                    print(f"Progress: {found}/{limit} | Current DRR: {current_drr:.2f}")

        except Exception as e:
            pass
        idx += 1

    return pd.DataFrame(results)

# EXECUTE LFW STUDY
lfw_final_results = run_lfw_benchmark(1000)

# FINAL REPORT
print("\n" + "="*50)
print("          LFW RESEARCH EVALUATION SUMMARY          ")
print("="*50)
print(f"Total Successful Samples:  {len(lfw_final_results)}")
print(f"Defense Recovery Rate:     {lfw_final_results['DRR'].mean()*100:.2f}%")
print(f"Attack Success Rate:       {lfw_final_results['ASR'].mean()*100:.2f}%")
print(f"Mean Threat Score (1-SSIM): {lfw_final_results['ThreatScore'].mean():.4f}")
print(f"Mean Inference Latency:    {lfw_final_results['Latency_ms'].mean():.2f}ms")
print("="*50)


--- [BENCHMARK] Dataset: LFW | Target: 1000 Samples ---
Progress: 50/1000 | Current DRR: 0.74
Progress: 100/1000 | Current DRR: 0.68
Progress: 150/1000 | Current DRR: 0.68
Progress: 200/1000 | Current DRR: 0.70
Progress: 250/1000 | Current DRR: 0.71
Progress: 300/1000 | Current DRR: 0.71
Progress: 350/1000 | Current DRR: 0.71
Progress: 400/1000 | Current DRR: 0.71
Progress: 450/1000 | Current DRR: 0.71
Progress: 500/1000 | Current DRR: 0.70
Progress: 550/1000 | Current DRR: 0.71
Progress: 600/1000 | Current DRR: 0.71
Progress: 650/1000 | Current DRR: 0.71
Progress: 700/1000 | Current DRR: 0.72
Progress: 750/1000 | Current DRR: 0.72
Progress: 800/1000 | Current DRR: 0.73
Progress: 850/1000 | Current DRR: 0.73
Progress: 900/1000 | Current DRR: 0.74
Progress: 950/1000 | Current DRR: 0.73
Progress: 1000/1000 | Current DRR: 0.73

          LFW RESEARCH EVALUATION SUMMARY          
Total Successful Samples:  1000
Defense Recovery Rate:     73.40%
Attack Success Rate:       92.70%
Mean Threa

In [8]:
from datasets import load_dataset
import pandas as pd
import time
import numpy as np
import cv2
from skimage.metrics import structural_similarity as ssim

def run_celeba_hq_research_benchmark(target_count=1000):
    print(f"\n--- [BENCHMARK] Dataset: CelebA-HQ (mattymchen) | Target: {target_count} Samples ---")
    results = []

    # 1. Initialize the specific Hugging Face dataset you provided
    # We use streaming=True to process 1,000 high-res images without using disk space
    try:
        ds = load_dataset("mattymchen/celeba-hq", split="train", streaming=True)
        print("Successfully connected to mattymchen/celeba-hq stream.")
    except Exception as e:
        print(f"Error connecting to dataset: {e}")
        return None

    found = 0
    print("Beginning Evaluation Pipeline (Detection -> Attack -> Defense)...")

    for sample in ds:
        if found >= target_count:
            break

        try:
            # 2. Extract and Convert Image
            # The dataset 'mattymchen/celeba-hq' provides PIL images under the 'image' key
            img_np = np.array(sample['image'].convert('RGB'))
            img = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR) # Convert to BGR for YOLO/GAN

            # Upscale/Resize to 512x512 for standard benchmark comparison
            img = cv2.resize(img, (512, 512))

            # 3. Baseline Detection (Must be visible initially)
            res_c = detector(img, verbose=False)
            if len(res_c[0].boxes) > 0:
                box = res_c[0].boxes.xyxy.cpu().numpy()[0]
                x1, y1, x2, y2 = map(int, box)

                # 4. Research Attack (High-Entropy Structured Patch)
                # This simulates the 'Nano-Banana' or 'Universal' evasion patterns
                atk_img = apply_structured_patch(img, x1, y1, x2-x1, y2-y1)
                res_a = detector(atk_img, verbose=False)

                # 5. Sentinel-DTP Defense (Proposed Reactive Pipeline)
                start_t = time.time()
                # Localization (Laplacian) -> Neutralization (Median) -> Purification (GFPGAN)
                purified, _ = sentinel_dtp_process(atk_img, [x1, y1, x2, y2])
                latency = (time.time() - start_t) * 1000

                # 6. Secure Re-Inference (Identity Verification)
                res_d = detector(purified, conf=0.15, verbose=False)

                # 7. Collect Metrics
                # SSIM compares Clean vs Purified to measure manifold restoration
                s_score = ssim(img, purified, channel_axis=2)

                results.append({
                    'ASR': 1 if len(res_a[0].boxes) == 0 else 0,
                    'DRR': 1 if len(res_d[0].boxes) > 0 else 0,
                    'ThreatScore': (1.0 - s_score) * 100,
                    'Latency_ms': latency
                })

                found += 1
                if found % 50 == 0:
                    avg_drr = pd.DataFrame(results)['DRR'].mean()
                    print(f"Progress: {found}/{target_count} | Current Recovery Rate: {avg_drr:.2f}")

        except Exception as e:
            # Skip any network timeout or corrupted image in the stream
            continue

    return pd.DataFrame(results)

# START RESEARCH EVALUATION
celeba_hq_results = run_celeba_hq_research_benchmark(1000)

# --- FINAL QUANTITATIVE REPORT ---
if celeba_hq_results is not None:
    print("\n" + "="*60)
    print("          CELEBA-HQ RESEARCH EVALUATION SUMMARY          ")
    print("="*60)
    print(f"Benchmark Size:            {len(celeba_hq_results)} High-Res Images")
    print(f"Defense Recovery Rate (DRR): {celeba_hq_results['DRR'].mean()*100:.2f}%")
    print(f"Attack Success Rate (ASR):   {celeba_hq_results['ASR'].mean()*100:.2f}%")
    print(f"Mean Threat Alert Score:     {celeba_hq_results['ThreatScore'].mean():.4f}")
    print(f"Mean Recovery Latency:       {celeba_hq_results['Latency_ms'].mean():.2f}ms")
    print("="*60)


--- [BENCHMARK] Dataset: CelebA-HQ (mattymchen) | Target: 1000 Samples ---


README.md:   0%|          | 0.00/539 [00:00<?, ?B/s]

Successfully connected to mattymchen/celeba-hq stream.
Beginning Evaluation Pipeline (Detection -> Attack -> Defense)...
Progress: 50/1000 | Current Recovery Rate: 0.98
Progress: 100/1000 | Current Recovery Rate: 0.98
Progress: 150/1000 | Current Recovery Rate: 0.99
Progress: 200/1000 | Current Recovery Rate: 0.99
Progress: 250/1000 | Current Recovery Rate: 0.99
Progress: 300/1000 | Current Recovery Rate: 0.99
Progress: 350/1000 | Current Recovery Rate: 0.99
Progress: 400/1000 | Current Recovery Rate: 0.99
Progress: 450/1000 | Current Recovery Rate: 0.99
Progress: 500/1000 | Current Recovery Rate: 0.99
Progress: 550/1000 | Current Recovery Rate: 0.99
Progress: 600/1000 | Current Recovery Rate: 0.99
Progress: 650/1000 | Current Recovery Rate: 0.99
Progress: 700/1000 | Current Recovery Rate: 0.99
Progress: 750/1000 | Current Recovery Rate: 0.99
Progress: 800/1000 | Current Recovery Rate: 0.99
Progress: 850/1000 | Current Recovery Rate: 0.99
Progress: 900/1000 | Current Recovery Rate: 0.9